# Imports

In [1]:
import behaviors
import no_signaling_sets
import numpy as np
import samplers

from tqdm import tqdm

In [2]:
delta = 2
m = 2

sampler = samplers.NoSignalingSampler(delta, m)
srns_set = no_signaling_sets.ShortRangeNoSignalingSet(delta, m)

# Run once

In [3]:
sampled_behavior = sampler.sample()

result = srns_set.lp_test(sampled_behavior)

print(f"alpha value: {-result.fun}")

2025-05-20 11:46:05.621 | SUCCESS  | samplers:sample_multiple:100 - Samples shape: (1, 32)


alpha value: 0.9716934625115148


## Check the closest SRNS behavior

In [4]:
yielded_behavior = behaviors.LatentSRNSBehavior(
    delta=delta,
    m=m,
    vector=np.clip(np.array(result.x[1:]), 0, 1),
)

print(f"Yielded behavior is {yielded_behavior}")
print(f"Yielded behavior is tested [{yielded_behavior.is_no_signaling()}] to being no signaling")


Yielded behavior is Behavior:
Short path (z=S):
[[0.24168849 0.3190209  0.3700062  0.16726666]
 [0.20192387 0.12459146 0.05414812 0.25688765]
 [0.22640597 0.36594022 0.09808826 0.51769446]
 [0.32998167 0.19044742 0.47775742 0.05815122]]
Long path (z=L) :
[[0.         0.19913831]
 [0.06590129 0.10142671]
 [0.23617638 0.        ]
 [0.14153469 0.1235893 ]
 [0.31785048 0.11871217]
 [0.03552542 0.        ]
 [0.20301174 0.43918812]
 [0.         0.0179454 ]]
------------
Yielded behavior is tested [True] to being no signaling


## Sanity check

In [5]:
print(f"PR box is tested [{srns_set.is_in_set(behaviors.pr_box)}] to being SRNS")


PR box is tested [False] to being SRNS


# Run on a batch and color the SRNS set

In [6]:
from time import time

delta = 2
m = 2

sampler = samplers.NoSignalingSampler(delta, m)
srns_set = no_signaling_sets.ShortRangeNoSignalingSet(delta, m)

n_samples = int(1e6)

In [ ]:
sample_arr = sampler.sample_multiple(number_of_samples=n_samples)
files_suffix = f"delta_{delta}_m_{m}_samples_{n_samples}_runtime_{time()}"

2025-05-20 11:46:12.471 | SUCCESS  | samplers:sample_multiple:100 - Samples shape: (1000000, 32)


In [ ]:
np.save(f"../data/view_srns/sampled_behaviors_{files_suffix}.npy", sample_arr)

In [ ]:
sample_list = list(sample_arr)

In [9]:
belonging_list = []
for i, vector in tqdm(enumerate(sample_list)):
    behavior = behaviors.RoutedBehavior(
        delta=delta,
        m=m,
        vector=vector,
    )
    belonging_list.append([i, srns_set.is_in_set(behavior)])

np.save(f"../data/view_srns/belonging_list_{files_suffix}.npy", belonging_list)

147162it [03:02, 805.86it/s]


KeyboardInterrupt: 

In [ ]:
analyzer = samplers.SamplesAnalyzer(
    delta=delta,
    m=m,
    sample_list=sample_list,
)

In [ ]:
analyzer.plot_projection(
    save_path=f"../data/view_srns/projection_{files_suffix}.png",
    plot=False,
    colors=belonging_list,
    )